In [ ]:
# NB_Log_Pipeline
# Called at end of pipeline to log the run summary

import notebookutils
from datetime import datetime
from pyspark.sql import Row
import uuid

# params = notebookutils.notebook.parameters
# status = params.get("status", "Success")

status = locals().get("status", "Success")

# Count what was processed this run
# files_processed   = int(params.get("files_processed", 0))
# files_quarantined = int(params.get("files_quarantined", 0))

files_processed = int(locals().get("files_processed", 0))
files_quarantined = int(locals().get("files_quarantined", 0))

# Count rows in Silver
total_rows = spark.table(
    "Blackwood_SilverLakehouse.silver_listings"
).count()

from pyspark.sql.functions import lit
from pyspark.sql.types import StringType

spark.createDataFrame([Row(
    run_id             = str(uuid.uuid4()),
    pipeline_name      = "PL_Blackwood_Wildcard",
    run_timestamp      = datetime.utcnow(),
    status             = status,
    files_found        = files_processed + files_quarantined,
    files_processed    = files_processed,
    files_quarantined  = files_quarantined,
    rows_loaded        = total_rows,
    duration_seconds   = 0,
    # Cast None to a String so Spark knows the column type
    error_message      = None, 
    notes              = f"Silver rows: {total_rows}"
)], schema="""
    run_id STRING, pipeline_name STRING, run_timestamp TIMESTAMP, 
    status STRING, files_found LONG, files_processed LONG, 
    files_quarantined LONG, rows_loaded LONG, duration_seconds LONG, 
    error_message STRING, notes STRING
""") \
.write.format("delta").mode("append") \
.saveAsTable("Blackwood_Bronze_Lakehouse.pipeline_log")


print(f"Pipeline logged: {status} | {files_processed} processed | {files_quarantined} quarantined")